# Comparing the PEG formulations — `polartox.peg_comparison`

The whole chain on a small example: **datagen → polarized_tree → pipeline → benchmark → peg_comparison**.

1. Generate two synthetic corpora with known ground truth (`AnnotatorPool`).
2. Search settings on them (`PolarizedTreesBenchmark`).
3. Take the best settings and keep **everything fixed except the PEG formulation** (`PEGComparison`): do the formulations recover the same dimensions, and do they build the same trees?

The corpora are small so that this runs in a minute; the paper's experiment is `benchmarks/notebooks/pegcomparison.ipynb`.

## 1. Two corpora with known ground truth

In [1]:
import tempfile
from pathlib import Path

import pandas as pd

from polartox import (
    AnnotatorPool, DEFAULT_DEPTH_WEIGHTS, DEFAULT_DIMENSIONS,
    PEGComparison, PolarizedTreesBenchmark, PolarizedTreesPipeline,
)

DIMS = list(DEFAULT_DIMENSIONS)
SCALE = 5

# Two corpora: the default intensities, and a weaker signal.
corpora = {}
for name, intensity in {"default": (0.3, 1.0), "weak": (0.2, 0.5)}.items():
    pool = AnnotatorPool(DEFAULT_DIMENSIONS, scale=SCALE, intensity_range=intensity,
                         depth_weights=DEFAULT_DEPTH_WEIGHTS, annotators_per_identity=3)
    corpora[name] = pool.generate_dataset(n_texts=30, seed=0)       # (data, ground_truth)
    print(name, corpora[name])

default GeneratedDataset(30 texts, 14580 rows)
   text_id  annotator_id gender politics  age education   orientation  rating
0        0             0   male     left  <25       low  heterosexual       5
1        0             1   male     left  <25       low  heterosexual       5
2        0             2   male     left  <25       low  heterosexual       5
3        0             3   male     left  <25       low        lgbtq+       5
4        0             4   male     left  <25       low        lgbtq+       5
weak GeneratedDataset(30 texts, 14580 rows)
   text_id  annotator_id gender politics  age education   orientation  rating
0        0             0   male     left  <25       low  heterosexual       5
1        0             1   male     left  <25       low  heterosexual       1
2        0             2   male     left  <25       low  heterosexual       5
3        0             3   male     left  <25       low        lgbtq+       1
4        0             4   male     left  <25      

## 2. Search settings

The benchmark takes one table of annotations and one ground truth, so the corpora are put together with distinct text ids; `text_groups` keeps them apart in the statistics. (`PEGComparison` takes the corpora as a dict `{name: (data, ground_truth)}` and keeps them apart itself.)

In [2]:
frames, ground_truth, text_groups = [], {}, {}
for offset, (name, (data, truth)) in zip((0, 1000), corpora.items()):
    frames.append(data.assign(text_id=data["text_id"] + offset))
    ground_truth.update({t + offset: entry for t, entry in truth.items()})
    text_groups.update({t + offset: name for t in truth})

benchmark = PolarizedTreesBenchmark(
    pipeline=PolarizedTreesPipeline(dims=DIMS, scale=SCALE, theta_filter=0.3, h=0.1, max_depth=4),
    annotations=pd.concat(frames, ignore_index=True),
    ground_truth=ground_truth,
    text_groups=text_groups,
    search_space={"theta_filter": [0.2, 0.3], "max_depth": [4, 6], "h": [0.1, 0.2],
                  "variant": ["max", "weighted", "harmonic"], "relative_h": [True], "theta_stop": [0.1]},
    strategy="random", n_runs=10, top_k=3, verbose=False,
).run()

benchmark.top_configs_[["theta_filter", "max_depth", "h", "variant", "jaccard", "exact_match"]].round(3)

,theta_filter,max_depth,h,variant,jaccard,exact_match
0,0.2,4,0.2,harmonic,0.742,0.428
1,0.2,4,0.1,weighted,0.738,0.429
2,0.3,6,0.2,harmonic,0.737,0.418


## 3. The same settings, every formulation

`PEGComparison.from_benchmark` takes the corpora, `dims` and `scale` from the benchmark. `run` takes the non-PEG settings of each chosen row (the row's own `variant` is ignored) and runs every formulation with them.

In [3]:
comparison = PEGComparison.from_benchmark(benchmark)
comparison.run(benchmark.top_configs_.head(2))      # settings labelled by the table's index

# Sanity check: with its own formulation, a chosen row reproduces what the benchmark saved.
print("numbers reproduced:", comparison.check_against_benchmark(benchmark.top_configs_))

numbers reproduced: 44


### Recovery and tree shape

One row per formulation, averaged over the corpora with equal weight.

In [4]:
comparison.overview()[["jaccard", "precision", "recall", "exact_match",
                       "mean_leaves", "mean_depth", "mean_leaf_size", "mean_residual_ndfu"]].round(3)

jaccard  precision  recall  exact_match  mean_leaves  \
setting variant                                                          
0       max         0.653      0.736   0.907        0.248       12.888   
        weighted    0.692      0.833   0.820        0.392        7.988   
        min         0.717      0.804   0.913        0.356       11.489   
        mean        0.728      0.806   0.922        0.409       11.356   
        harmonic    0.742      0.824   0.901        0.428       10.172   
1       max         0.651      0.734   0.907        0.248       13.085   
        weighted    0.738      0.831   0.907        0.429       10.918   
        min         0.717      0.804   0.913        0.356       11.563   
        mean        0.725      0.803   0.922        0.409       11.460   
        harmonic    0.709      0.791   0.918        0.356       11.191   

                  mean_depth  mean_leaf_size  mean_residual_ndfu  
setting variant                                                   
0       max            2.701          63.229               0.275  
        weighted       2.322         108.957               0.221  
        min            2.624          68.457               0.243  
        mean           2.610          68.722               0.252  
        harmonic       2.513          80.545               0.237  
1       max            2.714          62.690               0.279  
        weighted       2.564          74.073               0.239  
        min            2.627          68.299               0.244  
        mean           2.623          68.465               0.257  
        harmonic       2.595          69.004               0.245

### By the true number of active dimensions (k)

*Precision* low means extra dimensions were added, *recall* low means true ones were missed.

In [5]:
comparison.recovery_by_k().round(3)

texts  jaccard  precision  recall  exact_match
setting variant  k_true                                                
0       max      1           8    0.706      0.706   1.000        0.625
                 2          18    0.504      0.559   0.944        0.000
                 3          16    0.669      0.800   0.854        0.188
                 4          14    0.796      0.918   0.857        0.429
        weighted 1           8    0.844      0.844   1.000        0.750
                 2          18    0.630      0.713   0.917        0.111
                 3          16    0.786      0.964   0.812        0.625
                 4          14    0.575      0.843   0.589        0.286
        min      1           8    0.806      0.806   1.000        0.750
                 2          18    0.524      0.580   0.944        0.000
                 3          16    0.781      0.906   0.875        0.438
                 4          14    0.843      0.986   0.857        0.500
        mean     1           8    0.806      0.806   1.000        0.750
                 2          18    0.519      0.574   0.944        0.000
                 3          16    0.781      0.906   0.875        0.438
                 4          14    0.893      1.000   0.893        0.714
        harmonic 1           8    0.844      0.844   1.000        0.750
                 2          18    0.615      0.670   0.944        0.111
                 3          16    0.716      0.841   0.812        0.438
                 4          14    0.875      1.000   0.875        0.643
1       max      1           8    0.706      0.706   1.000        0.625
                 2          18    0.498      0.554   0.944        0.000
                 3          16    0.669      0.800   0.854        0.188
                 4          14    0.796      0.918   0.857        0.429
        weighted 1           8    0.844      0.844   1.000        0.750
                 2          18    0.550      0.606   0.944        0.056
                 3          16    0.786      0.953   0.833        0.562
                 4          14    0.861      0.986   0.875        0.571
        min      1           8    0.806      0.806   1.000        0.750
                 2          18    0.524      0.580   0.944        0.000
                 3          16    0.781      0.906   0.875        0.438
                 4          14    0.843      0.986   0.857        0.500
        mean     1           8    0.806      0.806   1.000        0.750
                 2          18    0.509      0.565   0.944        0.000
                 3          16    0.781      0.906   0.875        0.438
                 4          14    0.893      1.000   0.893        0.714
        harmonic 1           8    0.800      0.800   1.000        0.750
                 2          18    0.517      0.572   0.944        0.000
                 3          16    0.747      0.872   0.875        0.375
                 4          14    0.861      0.986   0.875        0.571

### Do the formulations build the same trees? (ARI)

1 = the same groups of annotators, about 0 = as different as chance.

In [6]:
setting = next(iter(comparison.settings_))
print(f"setting {setting}: mean ARI over the corpora")
comparison.ari_matrix(setting).round(3)

setting 0: mean ARI over the corpora


,max,weighted,min,mean,harmonic
max,1.000,0.603,0.693,0.817,0.752
weighted,0.603,1.000,0.686,0.718,0.747
min,0.693,0.686,1.000,0.836,0.780
mean,0.817,0.718,0.836,1.000,0.854
harmonic,0.752,0.747,0.780,0.854,1.000


In [7]:
comparison.ari_by_k().round(3)

setting  pair             k_true
0        max - weighted   1         0.765
                          2         0.684
                          3         0.553
                          4         0.479
         max - min        1         0.785
                                    ...  
1        mean - harmonic  4         0.850
         all pairs        1         0.887
                          2         0.837
                          3         0.784
                          4         0.780
Name: ari, Length: 88, dtype: float64

### The same text, different trees

In [8]:
ranking = comparison.disagreement()      # lowest mean ARI first
ranking.head(3)

,corpus,text_id,k_true,mean_ari
0,default,27,4,0.284566
1,weak,1027,4,0.375046
2,default,0,3,0.385593


In [9]:
worst = ranking.iloc[0]
comparison.show_text(worst["corpus"], worst["text_id"])

########## default, text 27 | true active dimensions: politics (alpha 0.62), orientation (alpha 0.71), age (alpha 0.70), gender (alpha 0.55)
             first split                     dimensions used  leaves  depth  Jaccard
formulation                                                                         
max             politics  age, gender, orientation, politics      18      3      1.0
weighted            none                                none       1      0      0.0
min          orientation  age, gender, orientation, politics      14      3      1.0
mean         orientation  age, gender, orientation, politics      14      3      1.0
harmonic     orientation  age, gender, orientation, politics      14      3      1.0
--- max ---
└── root (n=486, nDFU=0.450) split 'politics' (PEG=0.476)
    ├── politics=center (n=162, nDFU=0.926) split 'orientation' (PEG=0.581)
    │   ├── orientation=heterosexual (n=81, nDFU=0.345) split 'gender' (PEG=0.280)
    │   │   ├── gender=female (n=27

## 4. Save

`save` writes the tables as CSV files.

In [10]:
with tempfile.TemporaryDirectory() as folder:
    saved = comparison.save(Path(folder) / "peg_comparison")
    print(sorted(path.name for path in saved.iterdir()))

['ari_by_k.csv', 'ari_per_text_setting0_default.csv', 'ari_per_text_setting0_weak.csv', 'ari_per_text_setting1_default.csv', 'ari_per_text_setting1_weak.csv', 'ari_setting0_default.csv', 'ari_setting0_weak.csv', 'ari_setting1_default.csv', 'ari_setting1_weak.csv', 'depth_distribution_setting0.csv', 'depth_distribution_setting1.csv', 'dimension_usage_setting0.csv', 'dimension_usage_setting1.csv', 'fcp', 'overview.csv', 'per_run.csv', 'per_text.csv', 'recovery_by_k.csv', 'retention_curve.csv', 'selected_settings.csv', 'similarity_ari_all_settings.csv', 'similarity_ari_setting0.csv', 'similarity_ari_setting1.csv', 'similarity_dims_all_settings.csv', 'similarity_dims_setting0.csv', 'similarity_dims_setting1.csv', 'similarity_first_split_all_settings.csv', 'similarity_first_split_setting0.csv', 'similarity_first_split_setting1.csv', 'similarity_nmi_all_settings.csv', 'similarity_nmi_setting0.csv', 'similarity_nmi_setting1.csv']
